In [ ]:
from PIL import Image

import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
import torchvision.transforms as transforms
from CNN import ImageNeuralNetwork

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),  
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
train_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/train', transform=train_transform)
test_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/val', transform=test_transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=512, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

In [ ]:
image, label = train_data[0]
print(image.shape)  # torch.Size([3, 64, 64])
print(label)

In [ ]:
class_names = train_data.classes
print(len(class_names))  # 200

In [ ]:
num_epochs = 300
net = ImageNeuralNetwork(64, 4, 6, 200, 64, 0.3).to(device)
loss_function = nn.CrossEntropyLoss(label_smoothing = 0.1)
optimizer = optim.SGD(net.parameters(), lr = 0.1, momentum = 0.9, weight_decay = 1e-4, nesterov = True)
scheduler = CosineAnnealingLR(optimizer, T_max = num_epochs, eta_min = 1e-7)

In [ ]:
for epoch in range(num_epochs):
    print(f'Training epoch {epoch}...')
    
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f'Loss: {running_loss / len(train_loader):.4f}, LR: {current_lr:.6f}')

In [ ]:
correct = 0
total = 0

net.eval()
with torch.no_grad(): 
    for data in test_loader:
        images, labels = data
        
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}%')

In [ ]:
torch.save(net.state_dict(), f'trained_net_{accuracy}.pth')

In [ ]:
net = ImageNeuralNetwork(64, 4, 6, 200, 0.3).to(device)
net.load_state_dict(torch.load(f'trained_net_{accuracy}.pth'))
#net.load_state_dict(torch.load("trained_net_91.01.pth", map_location = device, weights_only = False))

In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [ ]:
def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

In [ ]:
image_paths = ['uni.jpg']
images = [load_image(img) for img in image_paths]
net.eval()
with torch.no_grad():
    for image in images:
        outputs = net(image.to(device))
        _, predicted = torch.max(outputs, 1)
        print(f'Prediction: {class_names[predicted.item()]}')